# Linguistic Features Evaluation Notebook

## Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 1: Setup & Install Dependencies

In [ ]:
%pip install transformers jsonlines tqdm
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 28.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import torch
from torch.utils.data import DataLoader
#to fix  name 'Dataset' is not defined in 3.1
from torch.utils.data import Dataset
from sklearn.metrics import classification_report
from tqdm import tqdm
import json
# add to fix error
import jsonlines
from collections import Counter
import numpy as np


import joblib
import spacy
from transformers import RobertaTokenizer
from transformers import RobertaModel

## Step 2: Load the Tokenizer, Scaler, and spaCy Model

In [ ]:
# Load tokenizer and spaCy model
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
nlp = spacy.load("en_core_web_md")

# Load scaler used during training
scaler = joblib.load("/content/drive/MyDrive/SemEval-2024 Task 8/models/scaler.pkl")

In [ ]:
# Define global sets
FUNCTION_WORDS = {'AUX', 'CCONJ', 'DET', 'INTJ', 'PART', 'PRON', 'ADP', 'SCONJ'}
CONTENT_WORDS = {'NOUN', 'VERB', 'ADJ', 'ADV', 'PROPN'}

# Predefine patterns and POS/DEP tags based on dataset
ALL_DEP_RELATIONS = nlp.pipe_labels['parser']
ALL_POS_TAGS = nlp.pipe_labels['tagger']

def extract_linguistic_features(text):
    doc = nlp(text)

    # 1. Dependency Relation Frequency (normalized)
    dep_counter = Counter([token.dep_ for token in doc])
    dep_freq = np.array([dep_counter.get(dep, 0) for dep in ALL_DEP_RELATIONS])
    dep_freq = dep_freq / (np.sum(dep_freq) + 1e-8)

    # 2. Tree Depth: Max token.depth
    tree_depth = max([token.head.i - token.i for token in doc if token.head != token]) if len(doc) > 1 else 0
    tree_depth = np.array([tree_depth])

    # 3. Specific syntactic pattern presence (e.g., nsubj -> ROOT -> dobj)
    pattern_present = any(
        token.dep_ == 'ROOT' and
        any(child.dep_ == 'nsubj' for child in token.children) and
        any(child.dep_ == 'dobj' for child in token.children)
        for token in doc
    )
    pattern_feature = np.array([1 if pattern_present else 0])

    # 4. POS tag bigram frequency (simplified)
    pos_bigrams = Counter()
    for i in range(len(doc) - 1):
        bigram = (doc[i].pos_, doc[i+1].pos_)
        pos_bigrams[bigram] += 1
    # Use a few common patterns for simplicity
    common_bigrams = [('NOUN', 'VERB'), ('VERB', 'NOUN'), ('ADJ', 'NOUN')]
    pos_bigram_feats = np.array([pos_bigrams.get(bigram, 0) for bigram in common_bigrams])

    # 5. Function word to content word ratio
    function_count = sum(1 for token in doc if token.pos_ in FUNCTION_WORDS)
    content_count = sum(1 for token in doc if token.pos_ in CONTENT_WORDS)
    func_content_ratio = np.array([function_count / (content_count + 1e-8)])

    # 6. Number of clauses per sentence (using "ccomp", "advcl", etc.)
    clause_tags = {"ccomp", "xcomp", "advcl", "relcl", "acl"}
    clause_count = sum(1 for token in doc if token.dep_ in clause_tags)
    clause_feat = np.array([clause_count])

    # 7. Head POS Tag Distribution
    head_pos_counter = Counter([token.head.pos_ for token in doc])
    common_heads = ['NOUN', 'VERB', 'ADJ']
    head_pos_feats = np.array([head_pos_counter.get(pos, 0) for pos in common_heads])

    # Combine all features
    features = np.concatenate([
        dep_freq,
        tree_depth,
        pattern_feature,
        pos_bigram_feats,
        func_content_ratio,
        clause_feat,
        head_pos_feats
    ])

    return features

## Step 3: Define Custom Text Dataset and the Model

### 🔹 3.1 Define Custom Text Dataset Class

In [ ]:
class CustomTextDataset(Dataset):
    def __init__(self, file_path, tokenizer, scaler, max_length=512):
        self.data = []
        self.max_length = max_length
        self.scaler = scaler

        with jsonlines.open(file_path) as reader:
            lines = list(reader)  # So tqdm knows total length
            for obj in tqdm(lines, desc=f"Processing {file_path}"):
                text = obj["text"]
                label = obj["label"]

                encoding = tokenizer(
                    text,
                    truncation=True,
                    padding='max_length',
                    max_length=max_length,
                    return_tensors='pt'
                )

                ling_feats = extract_linguistic_features(text)
                ling_feats = self.scaler.fit_transform(ling_feats.reshape(1, -1))

                self.data.append((encoding, ling_feats, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        encoding, ling_feats, label = self.data[idx]
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'ling_feats': torch.tensor(ling_feats, dtype=torch.float),
            'labels': torch.tensor(label, dtype=torch.long)
        }

### 🔹 3.2 Define the Model

In [ ]:
import torch.nn as nn

In [ ]:
class LinguisticRobertaClassifier(nn.Module):
    def __init__(self, ling_feat_dim, num_labels=2):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.roberta.config.hidden_size + ling_feat_dim, num_labels)  # concat CLS + linguistic features

    # def forward(self, input_ids, attention_mask, ling_feats):
    #     outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
    #     cls_emb = outputs.last_hidden_state[:, 0, :]  # CLS token
    #     combined = torch.cat((cls_emb, ling_feats), dim=1)
    #     combined = self.dropout(combined)
    #     logits = self.classifier(combined)
    #     return logits
    def forward(self, input_ids, attention_mask, ling_feats):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_emb = outputs.last_hidden_state[:, 0, :]  # CLS token
        ling_feats = ling_feats.squeeze(1)  # <- Fix here
        combined = torch.cat((cls_emb, ling_feats), dim=1)
        combined = self.dropout(combined)
        logits = self.classifier(combined)
        return logits

## Step 4: Load the Test Dataset

### 🔹 4.1 Create the Test Dataset

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Define test dataset path
test_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_monolingual.jsonl"

# Load dataset
# add scaler to fix TypeError: CustomTextDataset.__init__() missing 1 required positional argument: 'scaler'
test_dataset = CustomTextDataset(test_file, tokenizer, scaler)

ling_feat_dim = len(extract_linguistic_features("Example text"))

Processing /content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_monolingual.jsonl: 100%|██████████| 34272/34272 [29:31<00:00, 19.34it/s]


### 🔹 4.2 Save the Test Dataset

In [ ]:
# Save the dataset for the first time run
torch.save(test_dataset, "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/test_dataset_ver1.pt")

### 🔹 4.3 Load the Test Dataset

In [ ]:
ling_feat_dim = len(extract_linguistic_features("Example text"))

In [ ]:
test_dataset = torch.load(
    "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/test_dataset_ver1.pt",
    weights_only=False
)

# Create DataLoader
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

## Step 5: Load the Trained Model

In [ ]:
model_path = "/content/drive/MyDrive/SemEval-2024 Task 8/models/linguistic models/linguistic_model_ver1.pth"
model = LinguisticRobertaClassifier(ling_feat_dim)
model.load_state_dict(torch.load(model_path))
model.eval()

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


LinguisticRobertaClassifier(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              

## Step 6: Evaluate the Model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        ling_feats = batch['ling_feats'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, ling_feats=ling_feats)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Print classification report
print(classification_report(all_labels, all_preds, digits=4))

Evaluating: 100%|██████████| 2142/2142 [08:23<00:00,  4.25it/s]

              precision    recall  f1-score   support

           0     0.8171    0.7856    0.8010     16272
           1     0.8127    0.8410    0.8266     18000

    accuracy                         0.8147     34272
   macro avg     0.8149    0.8133    0.8138     34272
weighted avg     0.8148    0.8147    0.8145     34272

